## Introduction

In previous notebooks, we:
1. [Built a minimal VLM from scratch](2025-12-24-minimal-vlm-from-scratch.html) and trained it for image captioning
2. [Instruction fine-tuned it for object detection](2025-12-25-vlm-instruction-tuning-od.html) with JSON output

Now, let's tackle **Visual Question Answering (VQA)** - teaching the model to answer natural language questions about images.

### Examples

| Image | Question | Answer |
|-------|----------|--------|
| 🖼️ Photo of a cat | "What animal is this?" | "cat" |
| 🖼️ Beach scene | "What is the weather like?" | "sunny" |
| 🖼️ Kitchen | "How many people are there?" | "2" |

Unlike object detection (structured JSON), VQA requires **natural language answers** - short, direct responses to questions.

## Setup

In [ ]:
!uv pip install -q transformers datasets torch torchvision pillow accelerate einops timm

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    ViTModel,
    ViTImageProcessor,
)
from datasets import load_dataset
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import textwrap
import random
import os
import warnings
warnings.filterwarnings('ignore')

%config InlineBackend.figure_format = 'retina'

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Part 1: Load the Pretrained VLM (Caption Model)

We load from the **caption-trained model** (`mini-vlm-flickr8k/`), not the OD model. This is because:
- Caption model generates fluent natural language
- OD model is biased toward JSON output
- VQA needs natural language answers

In [ ]:
# Model architecture (same as previous notebooks)

class VisionProjector(nn.Module):
    """Projects vision features into the language model's embedding space."""
    
    def __init__(self, vision_dim: int, language_dim: int):
        super().__init__()
        self.projection = nn.Sequential(
            nn.Linear(vision_dim, language_dim),
            nn.GELU(),
            nn.LayerNorm(language_dim),
            nn.Linear(language_dim, language_dim),
        )
    
    def forward(self, vision_features: torch.Tensor) -> torch.Tensor:
        return self.projection(vision_features)


class MiniVLM(nn.Module):
    """A minimal Vision-Language Model."""
    
    def __init__(
        self,
        vision_encoder: ViTModel,
        language_model: AutoModelForCausalLM,
        projector: VisionProjector,
        tokenizer: AutoTokenizer,
    ):
        super().__init__()
        self.vision_encoder = vision_encoder
        self.language_model = language_model
        self.projector = projector
        self.tokenizer = tokenizer
        
        for param in self.vision_encoder.parameters():
            param.requires_grad = False
    
    def encode_image(self, pixel_values: torch.Tensor) -> torch.Tensor:
        with torch.no_grad():
            vision_outputs = self.vision_encoder(pixel_values=pixel_values)
        image_features = vision_outputs.last_hidden_state
        projected = self.projector(image_features)
        return projected
    
    def forward(
        self,
        pixel_values: torch.Tensor,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        labels: torch.Tensor = None,
    ):
        batch_size = pixel_values.shape[0]
        image_embeds = self.encode_image(pixel_values)
        num_image_tokens = image_embeds.shape[1]
        
        text_embeds = self.language_model.get_input_embeddings()(input_ids)
        combined_embeds = torch.cat([image_embeds, text_embeds], dim=1)
        
        image_attention = torch.ones(
            (batch_size, num_image_tokens),
            dtype=attention_mask.dtype,
            device=attention_mask.device
        )
        combined_attention = torch.cat([image_attention, attention_mask], dim=1)
        
        if labels is not None:
            image_labels = torch.full(
                (batch_size, num_image_tokens),
                fill_value=-100,
                dtype=labels.dtype,
                device=labels.device
            )
            combined_labels = torch.cat([image_labels, labels], dim=1)
        else:
            combined_labels = None
        
        outputs = self.language_model(
            inputs_embeds=combined_embeds,
            attention_mask=combined_attention,
            labels=combined_labels,
            return_dict=True,
        )
        
        return outputs
    
    @torch.no_grad()
    def generate(
        self,
        pixel_values: torch.Tensor,
        prompt: str,
        max_new_tokens: int = 20,
        temperature: float = 0.7,
        do_sample: bool = True,
    ) -> str:
        """Generate a response for an image given a prompt."""
        self.eval()
        
        image_embeds = self.encode_image(pixel_values)
        prompt_ids = self.tokenizer.encode(prompt, return_tensors="pt").to(pixel_values.device)
        generated_ids = prompt_ids.clone()
        
        for _ in range(max_new_tokens):
            current_embeds = self.language_model.get_input_embeddings()(generated_ids)
            full_embeds = torch.cat([image_embeds, current_embeds], dim=1)
            
            outputs = self.language_model(inputs_embeds=full_embeds)
            next_token_logits = outputs.logits[:, -1, :]
            
            if do_sample:
                probs = F.softmax(next_token_logits / temperature, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)
            else:
                next_token = next_token_logits.argmax(dim=-1, keepdim=True)
            
            generated_ids = torch.cat([generated_ids, next_token], dim=1)
            
            if next_token.item() == self.tokenizer.eos_token_id:
                break
        
        return self.tokenizer.decode(generated_ids[0], skip_special_tokens=True)

In [ ]:
# Model names
vision_model_name = "google/vit-base-patch16-224"
lm_model_name = "HuggingFaceTB/SmolLM-135M"
pretrained_dir = "mini-vlm-flickr8k"  # Caption model, NOT OD model

# Load base models
vision_encoder = ViTModel.from_pretrained(vision_model_name)
language_model = AutoModelForCausalLM.from_pretrained(lm_model_name)
tokenizer = AutoTokenizer.from_pretrained(lm_model_name)
image_processor = ViTImageProcessor.from_pretrained(vision_model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Create projector
vision_dim = vision_encoder.config.hidden_size
language_dim = language_model.config.hidden_size
projector = VisionProjector(vision_dim, language_dim)

# Load pretrained caption model weights
if os.path.exists(f"{pretrained_dir}/mini_vlm_full.pt"):
    print(f"Loading pretrained CAPTION model from {pretrained_dir}/")
    checkpoint = torch.load(f"{pretrained_dir}/mini_vlm_full.pt", map_location='cpu')
    projector.load_state_dict(checkpoint['projector_state_dict'])
    language_model.load_state_dict(checkpoint['language_model_state_dict'])
    print("Loaded pretrained caption model weights!")
else:
    print("No pretrained weights found. Starting from scratch.")
    print("(Run the captioning notebook first for better results)")

# Create VLM
vlm = MiniVLM(vision_encoder, language_model, projector, tokenizer)
vlm = vlm.to(device)

print(f"\nModel loaded on {device}")
print(f"Trainable parameters: {sum(p.numel() for p in vlm.parameters() if p.requires_grad):,}")

## Part 2: Load VQA Dataset

We'll use VQAv2 - one of the most popular visual question answering benchmarks. Each sample has:
- An image
- A question about the image
- Multiple human-provided answers (we'll use the most common one)

In [ ]:
# Load VQAv2 dataset (validation split - it's large enough)
# Using streaming to handle the large dataset
vqa_dataset_stream = load_dataset('lmms-lab/VQAv2', split='validation', streaming=True)

# Take a subset for training (convert stream to list)
num_samples = 2000
print(f"Loading {num_samples} samples from VQAv2...")

vqa_samples = []
for i, sample in enumerate(vqa_dataset_stream):
    if i >= num_samples:
        break
    vqa_samples.append(sample)
    if (i + 1) % 500 == 0:
        print(f"  Loaded {i + 1} samples...")

print(f"\nLoaded {len(vqa_samples)} VQA samples")
print(f"Sample keys: {vqa_samples[0].keys()}")

In [ ]:
# Look at a few samples
def get_most_common_answer(answers):
    """Get the most common answer from the list of annotator answers."""
    answer_counts = {}
    for ans in answers:
        a = ans['answer']
        answer_counts[a] = answer_counts.get(a, 0) + 1
    return max(answer_counts, key=answer_counts.get)

# Display a few examples
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, ax in enumerate(axes):
    sample = vqa_samples[i]
    image = sample['image']
    question = sample['question']
    answer = get_most_common_answer(sample['answers'])
    
    ax.imshow(image)
    ax.set_title(f"Q: {question}\nA: {answer}", fontsize=10, wrap=True)
    ax.axis('off')

plt.suptitle("VQAv2 Dataset Samples", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Part 3: Create VQA Instruction Dataset

We format each sample as:
- **Input:** `"Question: {question} Answer:"`
- **Target:** `"{answer}"`

The model learns to complete the answer after seeing the image and question.

In [ ]:
class VQADataset(Dataset):
    """Dataset for VQA instruction tuning."""
    
    def __init__(self, samples, image_processor, tokenizer, max_length=64):
        self.samples = samples
        self.image_processor = image_processor
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        
        # Process image
        image = sample['image'].convert('RGB')
        pixel_values = self.image_processor(image, return_tensors="pt").pixel_values.squeeze(0)
        
        # Get question and answer
        question = sample['question']
        answer = get_most_common_answer(sample['answers'])
        
        # Format: "Question: {q} Answer: {a}"
        prompt = f"Question: {question} Answer:"
        full_text = f"{prompt} {answer}"
        
        # Tokenize
        encoding = self.tokenizer(
            full_text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        input_ids = encoding['input_ids'].squeeze(0)
        attention_mask = encoding['attention_mask'].squeeze(0)
        
        # Create labels - mask the prompt part (only train on answer)
        prompt_tokens = self.tokenizer.encode(prompt, add_special_tokens=False)
        prompt_len = len(prompt_tokens)
        
        labels = input_ids.clone()
        labels[:prompt_len] = -100  # Don't compute loss on prompt
        labels[attention_mask == 0] = -100  # Don't compute loss on padding
        
        return {
            'pixel_values': pixel_values,
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': labels,
        }


# Create train/test split
train_samples = vqa_samples[:1800]
test_samples = vqa_samples[1800:]

train_dataset = VQADataset(train_samples, image_processor, tokenizer)
test_dataset = VQADataset(test_samples, image_processor, tokenizer)

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=0,
)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Number of batches: {len(train_loader)}")

In [ ]:
# Verify a batch
batch = next(iter(train_loader))
print(f"Batch pixel_values shape: {batch['pixel_values'].shape}")
print(f"Batch input_ids shape: {batch['input_ids'].shape}")

# Decode first sample
print(f"\nSample text:")
print(tokenizer.decode(batch['input_ids'][0], skip_special_tokens=True))

## Part 4: Test BEFORE Training

Let's see how the caption model responds to VQA questions before instruction tuning.

In [ ]:
def generate_vqa_answer(model, image, question, image_processor, device):
    """Generate an answer to a visual question."""
    model.eval()
    
    if not isinstance(image, Image.Image):
        image = Image.fromarray(image)
    image = image.convert('RGB')
    
    pixel_values = image_processor(image, return_tensors="pt").pixel_values.to(device)
    
    prompt = f"Question: {question} Answer:"
    
    response = model.generate(
        pixel_values,
        prompt=prompt,
        max_new_tokens=15,
        temperature=0.5,
        do_sample=True,
    )
    
    # Extract just the answer part
    if "Answer:" in response:
        answer = response.split("Answer:")[-1].strip()
    else:
        answer = response
    
    return answer, response

In [ ]:
# Test on a few samples BEFORE training
test_indices = [0, 3, 6, 9, 12]
test_data_for_comparison = []

print("=" * 70)
print("BEFORE VQA INSTRUCTION TUNING")
print("(Model was trained on captions, not Q&A)")
print("=" * 70)

before_answers = []
for idx in test_indices:
    sample = test_samples[idx]
    image = sample['image']
    question = sample['question']
    gt_answer = get_most_common_answer(sample['answers'])
    
    pred_answer, full_response = generate_vqa_answer(vlm, image, question, image_processor, device)
    before_answers.append(pred_answer)
    
    test_data_for_comparison.append({
        'image': image,
        'question': question,
        'gt_answer': gt_answer,
    })
    
    print(f"\nQ: {question}")
    print(f"  Model: {pred_answer[:50]}..." if len(pred_answer) > 50 else f"  Model: {pred_answer}")
    print(f"  GT: {gt_answer}")

## Part 5: VQA Instruction Fine-Tuning

Train the model to give short, direct answers to questions.

In [ ]:
def train_vlm_vqa(model, train_loader, num_epochs=8, lr=2e-4):
    """Train the VLM for VQA."""
    
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable_params, lr=lr)
    
    model.train()
    model.vision_encoder.eval()
    
    losses = []
    
    for epoch in range(num_epochs):
        epoch_loss = 0
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
        
        for batch in progress_bar:
            pixel_values = batch['pixel_values'].to(device)
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(
                pixel_values=pixel_values,
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels,
            )
            
            loss = outputs.loss
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
            optimizer.step()
            
            epoch_loss += loss.item()
            progress_bar.set_postfix({'loss': f"{loss.item():.4f}"})
        
        avg_loss = epoch_loss / len(train_loader)
        losses.append(avg_loss)
        print(f"Epoch {epoch+1} - Average Loss: {avg_loss:.4f}")
    
    return losses

In [ ]:
# Train for VQA
losses = train_vlm_vqa(vlm, train_loader, num_epochs=8, lr=2e-4)

In [ ]:
# Plot training loss
plt.figure(figsize=(8, 4))
plt.plot(range(1, len(losses)+1), losses, marker='o')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('VQA Instruction Fine-Tuning Loss')
plt.grid(True, alpha=0.3)
plt.show()

## Part 6: Test AFTER Training - Before vs After Comparison

In [ ]:
# Test on the same samples AFTER training
print("=" * 70)
print("COMPARISON: BEFORE vs AFTER VQA INSTRUCTION TUNING")
print("=" * 70)

after_answers = []
for i, data in enumerate(test_data_for_comparison):
    pred_answer, _ = generate_vqa_answer(vlm, data['image'], data['question'], image_processor, device)
    after_answers.append(pred_answer)
    
    print(f"\n{'='*70}")
    print(f"Q: {data['question']}")
    print(f"{'='*70}")
    print(f"BEFORE: {before_answers[i][:60]}..." if len(before_answers[i]) > 60 else f"BEFORE: {before_answers[i]}")
    print(f"AFTER:  {pred_answer}")
    print(f"GT:     {data['gt_answer']}")

In [ ]:
# Visual comparison
def wrap_text(text, width=30):
    return '\n'.join(textwrap.wrap(str(text), width=width))

fig, axes = plt.subplots(len(test_data_for_comparison), 1, figsize=(12, 4*len(test_data_for_comparison)))

for i, (data, before, after) in enumerate(zip(test_data_for_comparison, before_answers, after_answers)):
    ax = axes[i]
    ax.imshow(data['image'])
    
    title = f"Q: {data['question']}\n"
    title += f"BEFORE: {before[:40]}...\n" if len(before) > 40 else f"BEFORE: {before}\n"
    title += f"AFTER: {after}\n"
    title += f"GT: {data['gt_answer']}"
    
    # Color code: green if after matches GT, red otherwise
    color = 'green' if after.lower().strip() == data['gt_answer'].lower().strip() else 'black'
    ax.set_title(title, fontsize=10, color=color)
    ax.axis('off')

plt.suptitle("Before vs After VQA Training", fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## Part 7: Evaluate on More Test Samples

In [ ]:
# Evaluate on more test samples
num_eval = min(50, len(test_samples))
correct = 0
results = []

print(f"Evaluating on {num_eval} test samples...")

for i in tqdm(range(num_eval)):
    sample = test_samples[i]
    pred_answer, _ = generate_vqa_answer(vlm, sample['image'], sample['question'], image_processor, device)
    gt_answer = get_most_common_answer(sample['answers'])
    
    # Simple exact match (case-insensitive)
    is_correct = pred_answer.lower().strip() == gt_answer.lower().strip()
    if is_correct:
        correct += 1
    
    results.append({
        'question': sample['question'],
        'predicted': pred_answer,
        'ground_truth': gt_answer,
        'correct': is_correct
    })

accuracy = correct / num_eval * 100
print(f"\nExact Match Accuracy: {accuracy:.1f}% ({correct}/{num_eval})")

In [ ]:
# Show some correct and incorrect examples
print("\n" + "="*70)
print("CORRECT PREDICTIONS")
print("="*70)
for r in [x for x in results if x['correct']][:5]:
    print(f"Q: {r['question']}")
    print(f"A: {r['predicted']} (GT: {r['ground_truth']})")
    print()

print("\n" + "="*70)
print("INCORRECT PREDICTIONS")
print("="*70)
for r in [x for x in results if not x['correct']][:5]:
    print(f"Q: {r['question']}")
    print(f"Pred: {r['predicted']} | GT: {r['ground_truth']}")
    print()

## Part 8: Save the VQA Model

In [ ]:
# Save the VQA-tuned model
save_dir = "mini-vlm-vqa"
os.makedirs(save_dir, exist_ok=True)

torch.save({
    'projector_state_dict': vlm.projector.state_dict(),
    'language_model_state_dict': vlm.language_model.state_dict(),
    'config': {
        'vision_model_name': vision_model_name,
        'lm_model_name': lm_model_name,
        'vision_dim': vision_dim,
        'language_dim': language_dim,
    },
}, f"{save_dir}/mini_vlm_vqa.pt")

tokenizer.save_pretrained(f"{save_dir}/tokenizer")
image_processor.save_pretrained(f"{save_dir}/image_processor")

print(f"Model saved to {save_dir}/")
print(f"Contents: {os.listdir(save_dir)}")

## Summary

We successfully instruction fine-tuned our VLM for Visual Question Answering:

### What We Did

1. **Loaded the caption-trained VLM** (not the OD model - different task)
2. **Used VQAv2 dataset** with 1800 training samples
3. **Created Q&A format**: `"Question: {q} Answer: {a}"`
4. **Fine-tuned for 8 epochs** to learn short, direct answers
5. **Evaluated** with exact match accuracy

### Key Insights

- **Task-specific tuning matters**: Caption model → VQA model (not OD → VQA)
- **Short answers**: VQA typically requires 1-3 word answers
- **Prompt format**: `"Question: ... Answer:"` helps structure the task
- **Label masking**: Only train on the answer portion

### Model Family Tree

```
Base Models (ViT + SmolLM)
    │
    └── Caption Training (Flickr8k)
            │
            ├── OD Instruction Tuning → mini-vlm-od/
            │
            └── VQA Instruction Tuning → mini-vlm-vqa/  ← This notebook
```

### Limitations

- Small model (135M LLM) limits reasoning ability
- Exact match accuracy is strict ("2" vs "two" counted as wrong)
- Limited training data (1800 samples)
- No complex reasoning or counting abilities

### Next Steps

1. **Multi-task model**: Combine caption, OD, and VQA in one model
2. **Better evaluation**: Use VQA accuracy metric (considers answer variations)
3. **Larger models**: Try SmolLM-360M or 1.7B for better reasoning
4. **Chain-of-thought**: Add reasoning before final answer

## References

- [Building a Minimal VLM from Scratch](2025-12-24-minimal-vlm-from-scratch.html)
- [VLM Instruction Tuning for Object Detection](2025-12-25-vlm-instruction-tuning-od.html)
- [VQAv2 Dataset](https://visualqa.org/)
- [VQAv2 on Hugging Face](https://huggingface.co/datasets/lmms-lab/VQAv2)
- [LLaVA: Visual Instruction Tuning](https://llava-vl.github.io/)